In [ ]:
# Cell 1: import libraries
from pathlib import Path
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from concurrent.futures import ThreadPoolExecutor
from statsmodels.gam.api import GLMGam, BSplines

from patsy.builtins import bs
from scipy.special import gammaln
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from pygam import PoissonGAM, s, f as gam_f

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
np.set_printoptions(suppress=True, linewidth=140)



# Session 1 Data load, scope, and feature selection

In [ ]:
# Cell 2: load data
DATA_PATH = Path("script/data/parquet/full.parquet")
raw = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(raw):,} rows and {raw.shape[1]:,} columns from {DATA_PATH}.")



In [ ]:
# Cell 3: Toggles
PREDICTION_BASIS = "amount"  # toggle this between "count" and "amount" to switch the prediction target.

# Data scope
ISSUE_AGE_MIN = 0
ISSUE_AGE_MAX = 17
ATTAINED_AGE_MIN = 1
ATTAINED_AGE_MAX = 67
DURATION_MIN = 1
DURATION_MAX = 119

SMOKER_STATUS_KEEP = ["U"]
INSURANCE_PLAN_DROP = ["Other"]
POST_LEVEL_TERM_DROP = ["PLT"]



In [ ]:
# Cell 4: toggle variables
TARGET_CONFIG = {
    "count": {
        "actual_col": "Death_Count",
        "expected_col": "ExpDth_VBT2015_Cnt",
        "z_weight_col": "ExpDth_VBT2015_Cnt",
        "z_weight_label": "expected death count",
        "min_z_weight": 3.0,
        "exposure_col": "Policies_Exposed",
        "predicted_col": "Predicted_Deaths",
        "target_label": "Death Count",
        "rate_denominator_label": "policies exposed",
        "actual_summary_col": "Actual_Deaths",
        "expected_summary_col": "Expected_VBT",
        "predicted_summary_col": "Predicted_Deaths",
        "log_smoothing": 0.25,
    },
    "amount": {
        "actual_col": "Death_Claim_Amount",
        "expected_col": "ExpDth_VBT2015_Amt",
        "z_weight_col": "ExpDth_VBT2015_Cnt",
        "z_weight_label": "expected death count",
        "min_z_weight": 3.0,
        "exposure_col": "Amount_Exposed",
        "predicted_col": "Predicted_Death_Amount",
        "target_label": "Death Claim Amount",
        "rate_denominator_label": "amount exposed",
        "actual_summary_col": "Actual_Death_Amount",
        "expected_summary_col": "Expected_VBT_Amount",
        "predicted_summary_col": "Predicted_Death_Amount",
        "log_smoothing": 0.25,
    },
}
if PREDICTION_BASIS not in TARGET_CONFIG:
    raise ValueError(f"PREDICTION_BASIS must be one of {sorted(TARGET_CONFIG)}")


MODEL_CONFIG = TARGET_CONFIG[PREDICTION_BASIS]
MODEL_ACTUAL_COL = MODEL_CONFIG["actual_col"]
MODEL_EXPECTED_COL = MODEL_CONFIG["expected_col"]
MODEL_Z_WEIGHT_COL = MODEL_CONFIG["z_weight_col"]
MODEL_Z_WEIGHT_LABEL = MODEL_CONFIG["z_weight_label"]
MIN_Z_WEIGHT_FOR_REVIEW = float(MODEL_CONFIG["min_z_weight"])
MODEL_EXPOSURE_COL = MODEL_CONFIG["exposure_col"]
MODEL_PREDICTED_COL = MODEL_CONFIG["predicted_col"]
MODEL_TARGET_LABEL = MODEL_CONFIG["target_label"]
MODEL_RATE_DENOMINATOR_LABEL = MODEL_CONFIG["rate_denominator_label"]
MODEL_ACTUAL_SUMMARY_COL = MODEL_CONFIG["actual_summary_col"]
MODEL_EXPECTED_SUMMARY_COL = MODEL_CONFIG["expected_summary_col"]
MODEL_PREDICTED_SUMMARY_COL = MODEL_CONFIG["predicted_summary_col"]
MODEL_LOG_SMOOTHING = MODEL_CONFIG["log_smoothing"]

In [5]:
# Cell 5: apply filters, scopes, and groupings
OBSERVATION_YEARS = sorted(raw["Observation_Year"].dropna().unique())
AGE_IND_KEEP = sorted(raw["Age_Ind"].dropna().unique())
SEX_KEEP = sorted(raw["Sex"].dropna().unique())

METRIC_COLS = ["Death_Count", "ExpDth_VBT2015_Cnt", "Death_Claim_Amount", "ExpDth_VBT2015_Amt", "Policies_Exposed", "Amount_Exposed"]
MODEL_FEATURES = ["Sex", "Sex_M", "Attained_Age_mod", "Sex_x_Attained_Age_mod"]
ML_FEATURES = ["Sex_M", "Attained_Age_mod", "Sex_x_Attained_Age_mod"]
SCREEN_NUMERIC_FEATURES = [
    "Observation_Year", "Attained_Age_mod", "Sex_x_Attained_Age_mod", "Issue_Age_mod", "Duration",
]
SCREEN_CATEGORICAL_FEATURES = [
    "Age_Ind", "Sex", "Insurance_Plan_Group", "Face_Band_Group",
]
SCREEN_FEATURES = SCREEN_NUMERIC_FEATURES + SCREEN_CATEGORICAL_FEATURES
SCREEN_FEATURE_PARENT_ORDER = sorted(SCREEN_FEATURES, key=len, reverse=True)

df = raw.copy()

for col in [
    "Observation_Year", "Issue_Age", "Duration", "Issue_Year", "Attained_Age",
    "Amount_Exposed", "Policies_Exposed", "Death_Claim_Amount", "Death_Count",
    "ExpDth_VBT2015_Cnt", "ExpDth_VBT2015_Amt",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["Age_Ind", "Sex", "Smoker_Status", "Insurance_Plan", "Face_Amount_Band", "SOA_Post_Lvl_Ind"]:
    df[col] = df[col].astype("string").fillna("Missing")

df = df[
    df["Observation_Year"].isin(OBSERVATION_YEARS)
    & df["Issue_Age"].between(ISSUE_AGE_MIN, ISSUE_AGE_MAX)
    & df["Attained_Age"].between(ATTAINED_AGE_MIN, ATTAINED_AGE_MAX)
    & df["Duration"].between(DURATION_MIN, DURATION_MAX)
    & df["Age_Ind"].isin(AGE_IND_KEEP)
    & df["Sex"].isin(SEX_KEEP)
    & df["Smoker_Status"].isin(SMOKER_STATUS_KEEP)
    & ~df["Insurance_Plan"].isin(INSURANCE_PLAN_DROP)
    & ~df["SOA_Post_Lvl_Ind"].isin(POST_LEVEL_TERM_DROP)
].copy()

cap = df["Face_Amount_Band"].isin([
    "08: 1,000,000 - 2,499,999",
    "09: 2,500,000 - 4,999,999",
    "10: 5,000,000 - 9,999,999",
    "11: 10,000,000+",
])
df.loc[cap, "Face_Amount_Band"] = "08: 1,000,000+"
df.loc[cap, "Death_Claim_Amount"] = 1_000_000 * df.loc[cap, "Death_Count"]
df.loc[cap, "Amount_Exposed"] = 1_000_000 * df.loc[cap, "Policies_Exposed"]
df.loc[cap, "ExpDth_VBT2015_Amt"] = 1_000_000 * df.loc[cap, "ExpDth_VBT2015_Cnt"]

# Age modification
ISSUE_AGE_MOD_COL = "Issue_Age_mod"
ATTAINED_AGE_MOD_COL = "Attained_Age_mod"
ISSUE_AGE_MOD_MAX = ISSUE_AGE_MAX + 0.5
ATTAINED_AGE_MOD_MAX = ATTAINED_AGE_MAX + 0.5

df[ISSUE_AGE_MOD_COL] = df["Issue_Age"]
df[ATTAINED_AGE_MOD_COL] = df["Attained_Age"]
df.loc[df["Age_Ind"] == "1", ISSUE_AGE_MOD_COL] = df.loc[df["Age_Ind"] == "1", "Issue_Age"] + 0.5
df.loc[df["Age_Ind"] == "1", ATTAINED_AGE_MOD_COL] = df.loc[df["Age_Ind"] == "1", "Attained_Age"] + 0.5

scale = np.where(df["Issue_Year"].between(2014, 2019), 7 / (2019 - df["Issue_Year"] + 1).clip(lower=1), 1.0)
df["Policies_Exposed"] = df["Policies_Exposed"] * scale
df["Amount_Exposed"] = df["Amount_Exposed"] * scale

df["Issue_Age_mod_spline"] = df[ISSUE_AGE_MOD_COL].clip(np.nextafter(ISSUE_AGE_MIN, np.inf), np.nextafter(ISSUE_AGE_MOD_MAX, -np.inf))
df["Attained_Age_mod_spline"] = df[ATTAINED_AGE_MOD_COL].clip(np.nextafter(ATTAINED_AGE_MIN, np.inf), np.nextafter(ATTAINED_AGE_MOD_MAX, -np.inf))
df["Sex_M"] = (df["Sex"].astype(str) == "M").astype(float)
df["Sex_x_Attained_Age_mod"] = df["Sex_M"] * df[ATTAINED_AGE_MOD_COL]
df["Insurance_Plan_Group"] = df["Insurance_Plan"].replace({"UL": "Perm", "ULSG": "Perm", "VL": "Perm", "VLSG": "Perm"})
df["Face_Band_Group"] = np.where(df["Face_Amount_Band"].str.slice(0, 2).isin(["01", "02", "03", "04"]), "01-04: <100k", "05-11: >=100k")

df = df[
    (df[MODEL_EXPECTED_COL] > 0)
    & (df[MODEL_EXPOSURE_COL] > 0)
].copy()

model_data = df.groupby(MODEL_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
screen_data = df.groupby(SCREEN_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
model_data["Attained_Age_mod_spline"] = model_data[ATTAINED_AGE_MOD_COL].clip(np.nextafter(ATTAINED_AGE_MIN, np.inf), np.nextafter(ATTAINED_AGE_MOD_MAX, -np.inf))
model_data["Attained_Age_Band"] = pd.cut(model_data[ATTAINED_AGE_MOD_COL], list(range(0, 75, 5)), right=False, include_lowest=True).astype(str)

train_raw = df.copy()
holdout_raw = df.copy()
train_data = model_data.copy()
holdout_data = model_data.copy()
model_results = []
last_model = {}

scope = pd.DataFrame({
    "Rows": [len(raw), len(df), len(model_data)],
    MODEL_ACTUAL_SUMMARY_COL: [raw[MODEL_ACTUAL_COL].sum(), df[MODEL_ACTUAL_COL].sum(), model_data[MODEL_ACTUAL_COL].sum()],
    MODEL_EXPOSURE_COL: [raw[MODEL_EXPOSURE_COL].sum(), df[MODEL_EXPOSURE_COL].sum(), model_data[MODEL_EXPOSURE_COL].sum()],
}, index=["Raw", "Scoped", "Grouped by sex and attained age"])

target_settings = pd.DataFrame([{
    "Prediction_Basis": PREDICTION_BASIS,
    "Actual_Column": MODEL_ACTUAL_COL,
    "Expected_Column": MODEL_EXPECTED_COL,
    "Z_Weight_Column": MODEL_Z_WEIGHT_COL,
    "Exposure_Column": MODEL_EXPOSURE_COL,
    "Predicted_Column": MODEL_PREDICTED_COL,
}])
display(target_settings)
display(scope)


,Prediction_Basis,Actual_Column,Expected_Column,Z_Weight_Column,Exposure_Column,Predicted_Column
0,amount,Death_Claim_Amount,ExpDth_VBT2015_Amt,ExpDth_VBT2015_Cnt,Amount_Exposed,Predicted_Death_Amount


,Rows,Actual_Death_Amount,Amount_Exposed
Raw,3598960,3686564558,"3,828,939,427,052.111328"
Scoped,1424614,1930532965,"2,789,715,009,502.771484"
Grouped by sex and attained age,134,1930532965,"2,789,715,009,502.771973"


In [9]:
# Cell 6: helper functions

def search_knots(
    data,
    sex_value=None,
    family=sm.families.Poisson(),
    min_knots=1,
    max_knots=3,
    min_sep=8,
    boundary_gap=6,
    raw_candidates=None,
    search_alphas=None,
    final_alphas=None,
    n_workers=4,
    z_thresh=2.0,
    min_expected=None,
    verbose=True,
):
    z_weight_floor = MIN_Z_WEIGHT_FOR_REVIEW if min_expected is None else float(min_expected)
    lower = float(ATTAINED_AGE_MIN)
    upper = float(ATTAINED_AGE_MOD_MAX)
    search_grid = list(np.logspace(-1, 2.7, 3) if search_alphas is None else search_alphas)
    final_grid = list(np.logspace(-2, 2.7, 15) if final_alphas is None else final_alphas)
    candidates_raw = sorted(list(range(1, 35, 1)) + list(range(35, 67, 5)) if raw_candidates is None else raw_candidates)
    candidates = [k for k in candidates_raw if k > lower + boundary_gap and k < upper - boundary_gap]

    if sex_value is None:
        cells = data.groupby(["Attained_Age_mod_spline", "Sex"], dropna=False)[METRIC_COLS].sum().reset_index()
        cells = cells[cells[MODEL_EXPOSURE_COL] > 0].reset_index(drop=True)
        label = "combined sex"
    else:
        cells = data[data["Sex"].astype(str) == str(sex_value)].groupby(["Attained_Age_mod_spline"], dropna=False)[METRIC_COLS].sum().reset_index()
        cells = cells[cells[MODEL_EXPOSURE_COL] > 0].reset_index(drop=True)
        label = f"Sex={sex_value}"

    if verbose:
        print(f"Knot search for {label}")
        print(f"  min_knots={min_knots}  max_knots={max_knots}  min_sep={min_sep}  boundary_gap={boundary_gap}")
        print(f"  z_thresh={z_thresh}")
        print(f"  z_weight={MODEL_Z_WEIGHT_COL}  floor={z_weight_floor:g}")
        print(f"  candidates={len(candidates)}  age_range={min(candidates)} to {max(candidates)}")
        print(f"  cells={len(cells):,}  {MODEL_TARGET_LABEL.lower()}={cells[MODEL_ACTUAL_COL].sum():,.0f}")

    class CombinedSexPenalizedGAM:
        def __init__(self, beta, bs_obj, n_basis):
            self.params = np.asarray(beta, dtype=float)
            self._bs = bs_obj
            self._n_basis = int(n_basis)

        def predict(self, frame, offset=None):
            frame = frame.reset_index(drop=True)
            sex = frame["Sex"].astype(str).values
            is_male = (sex == "M").astype(float)
            is_female = 1.0 - is_male
            basis = self._bs.transform(frame[["Attained_Age_mod_spline"]].astype(float).values)
            design = np.column_stack([np.ones(len(frame)), is_male, basis * is_female[:, None], basis * is_male[:, None]])
            eta = design @ self.params
            if offset is not None:
                eta = eta + np.asarray(offset, dtype=float)
            return np.exp(np.clip(eta, -30, 30))

    def fit_combined(knots, alpha):
        y = cells[MODEL_ACTUAL_COL].astype(float).values
        offset = np.log(cells[MODEL_EXPOSURE_COL].astype(float).clip(1e-10).values)
        sex = cells["Sex"].astype(str).values
        is_male = (sex == "M").astype(float)
        is_female = 1.0 - is_male
        try:
            bs_obj = BSplines(
                cells[["Attained_Age_mod_spline"]],
                df=[len(knots) + 4],
                degree=[3],
                include_intercept=False,
                knot_kwds=[{"knots": list(knots), "lower_bound": lower, "upper_bound": upper}],
            )
        except Exception:
            return None
        basis = bs_obj.basis
        n_basis = basis.shape[1]
        design = np.column_stack([np.ones(len(cells)), is_male, basis * is_female[:, None], basis * is_male[:, None]])
        penalty = np.zeros((design.shape[1], design.shape[1]))
        spline_penalty = bs_obj.penalty_matrices[0]
        penalty[2:2 + n_basis, 2:2 + n_basis] = float(alpha) * spline_penalty
        penalty[2 + n_basis:design.shape[1], 2 + n_basis:design.shape[1]] = float(alpha) * spline_penalty
        beta = np.zeros(design.shape[1])
        for _ in range(50):
            eta = design @ beta + offset
            mu = np.exp(np.clip(eta, -30, 30))
            xtwx = (design * mu[:, None]).T @ design
            xtwz = (design * mu[:, None]).T @ (eta - offset + (y - mu) / mu.clip(1e-15))
            try:
                beta_new = np.linalg.solve(xtwx + penalty, xtwz)
            except np.linalg.LinAlgError:
                return None
            if np.max(np.abs(beta_new - beta)) < 1e-7:
                beta = beta_new
                break
            beta = beta_new
        mu_final = np.exp(np.clip(design @ beta + offset, -30, 30))
        ll = float(np.sum(y * np.log(mu_final.clip(1e-15)) - mu_final - gammaln(y + 1)))
        xtwxf = (design * mu_final[:, None]).T @ design
        try:
            edf = float(np.trace(np.linalg.solve(xtwxf + penalty, xtwxf)))
        except np.linalg.LinAlgError:
            edf = float(design.shape[1])
        return {
            "beta": beta,
            "wrapper": CombinedSexPenalizedGAM(beta, bs_obj, n_basis),
            "aic": -2.0 * ll + 2.0 * edf,
            "alpha": float(alpha),
            "edf": edf,
            "knots": list(knots),
        }

    def fit_per_sex(knots, alpha):
        y = cells[MODEL_ACTUAL_COL].astype(float)
        offset = np.log(cells[MODEL_EXPOSURE_COL].astype(float).clip(1e-10))
        exog = pd.DataFrame({"const": np.ones(len(cells))})
        try:
            bs_obj = BSplines(
                cells[["Attained_Age_mod_spline"]],
                df=[len(knots) + 4],
                degree=[3],
                include_intercept=False,
                knot_kwds=[{"knots": list(knots), "lower_bound": lower, "upper_bound": upper}],
            )
            result = GLMGam(y, exog=exog, smoother=bs_obj, alpha=[alpha], family=family, offset=offset).fit(disp=False)
        except Exception:
            return None
        edf_value = getattr(result, "edf", len(result.params))
        edf_value = float(np.sum(edf_value)) if np.ndim(edf_value) else float(edf_value)
        return {"result": result, "smoother": bs_obj, "aic": float(result.aic), "alpha": float(alpha), "edf": edf_value}

    def best_fit(knots, alpha_grid):
        best = None
        best_aic = np.inf
        for alpha in alpha_grid:
            fit = fit_per_sex(knots, alpha) if sex_value is not None else fit_combined(knots, alpha)
            if fit is not None and np.isfinite(fit["aic"]) and fit["aic"] < best_aic:
                best = fit
                best_aic = fit["aic"]
        return best

    def predict_fit(fit):
        offset = np.log(cells[MODEL_EXPOSURE_COL].astype(float).clip(1e-10))
        if sex_value is None:
            return fit["wrapper"].predict(cells, offset=offset.values)
        basis = fit["smoother"].transform(cells[["Attained_Age_mod_spline"]].to_numpy(dtype=float))
        design = np.column_stack([np.ones((len(cells), 1)), basis])
        return np.exp(design @ np.asarray(fit["result"].params, dtype=float) + offset.values)

    def z_score(fit):
        pred = predict_fit(fit)
        agg = pd.DataFrame({
            "age": cells["Attained_Age_mod_spline"].values,
            "actual": cells[MODEL_ACTUAL_COL].values.astype(float),
            "expected": cells[MODEL_EXPECTED_COL].values.astype(float),
            "z_weight": cells[MODEL_Z_WEIGHT_COL].values.astype(float),
            "predicted": pred,
        }).groupby("age").sum()
        credible = agg[agg["z_weight"] >= z_weight_floor]
        if len(credible) == 0:
            return (np.inf, np.inf, np.inf)
        ap_ratio = credible["actual"] / credible["predicted"].clip(1e-10)
        z_values = (ap_ratio - 1.0).abs() * np.sqrt(credible["z_weight"].clip(1.0))
        return (int((z_values > z_thresh).sum()), float(z_values.max()), float((np.maximum(z_values - z_thresh, 0) ** 2).sum()))

    rows = []
    best_score = (np.inf, np.inf, np.inf)
    best_knots = None
    best_n_knots = None

    for n_knots in range(int(min_knots), int(max_knots) + 1):
        knots = []
        remaining = list(candidates)
        score = (np.inf, np.inf, np.inf)
        eval_count = 0

        for step in range(n_knots):
            snapshot = list(knots)

            def try_add(candidate, selected=snapshot):
                if not all(abs(float(candidate) - float(knot)) >= float(min_sep) for knot in selected):
                    return None
                trial_knots = tuple(sorted(selected + [candidate]))
                fit = best_fit(list(trial_knots), search_grid)
                if fit is None:
                    return None
                return z_score(fit), candidate, fit

            with ThreadPoolExecutor(max_workers=int(n_workers)) as pool:
                step_results = [result for result in pool.map(try_add, remaining) if result is not None]
            eval_count += len(step_results)
            if not step_results:
                break
            step_results.sort(key=lambda item: item[0])
            score, candidate, fit = step_results[0]
            knots = sorted(knots + [candidate])
            remaining.remove(candidate)
            rows.append({"Stage": "add", "N_Knots": n_knots, "Step": step + 1, "Knots": list(knots), "Candidate": candidate, "Z_Violations": score[0], "Max_Z": score[1], "Excess_Z_Penalty": score[2], "AIC": fit["aic"], "Alpha": fit["alpha"], "EDF": fit.get("edf", np.nan)})
            if verbose:
                print(f"    step {step + 1}: add {candidate} -> knots={knots}  z_viol={score[0]}  max_z={score[1]:.3f}")

        if not knots:
            continue

        improved = True
        while improved:
            improved = False
            for old_knot in list(knots):
                base_knots = [k for k in knots if k != old_knot]
                swap_pool = [candidate for candidate in candidates if candidate not in knots and all(abs(float(candidate) - float(knot)) >= float(min_sep) for knot in base_knots)]

                def try_swap(candidate, base=base_knots):
                    trial_knots = sorted(base + [candidate])
                    fit = best_fit(trial_knots, search_grid)
                    if fit is None:
                        return None
                    return z_score(fit), trial_knots, candidate, fit

                with ThreadPoolExecutor(max_workers=int(n_workers)) as pool:
                    swap_results = [result for result in pool.map(try_swap, swap_pool) if result is not None]
                eval_count += len(swap_results)
                for swap_score, trial_knots, candidate, fit in swap_results:
                    if swap_score < score:
                        score = swap_score
                        knots = trial_knots
                        improved = True
                        rows.append({"Stage": "swap", "N_Knots": n_knots, "Step": np.nan, "Knots": list(knots), "Candidate": candidate, "Replaced": old_knot, "Z_Violations": score[0], "Max_Z": score[1], "Excess_Z_Penalty": score[2], "AIC": fit["aic"], "Alpha": fit["alpha"], "EDF": fit.get("edf", np.nan)})
                        if verbose:
                            print(f"    swap {old_knot} -> {candidate}  knots={knots}  z_viol={score[0]}  max_z={score[1]:.3f}")

        final_fit = best_fit(knots, final_grid)
        final_score = z_score(final_fit)
        rows.append({"Stage": "final", "N_Knots": n_knots, "Step": np.nan, "Knots": list(knots), "Z_Violations": final_score[0], "Max_Z": final_score[1], "Excess_Z_Penalty": final_score[2], "AIC": final_fit["aic"], "Alpha": final_fit["alpha"], "EDF": final_fit.get("edf", np.nan), "Evaluations": eval_count})
        if verbose:
            print(f"  n_knots={n_knots}: knots={knots}  gaps={[knots[i + 1] - knots[i] for i in range(len(knots) - 1)]}  z_viol={final_score[0]}  max_z={final_score[1]:.3f}")

        if score < best_score:
            best_score = score
            best_knots = knots
            best_n_knots = n_knots
        if best_score[0] == 0 and best_score[1] < 1.5:
            break

    final_fit = best_fit(best_knots, final_grid)
    final_score = z_score(final_fit)
    if sex_value is None:
        parameter_count = int(len(final_fit["beta"]))
    else:
        parameter_count = int(len(final_fit["result"].params))
    summary = {
        "Knots": list(best_knots),
        "N_Knots": int(best_n_knots),
        "Alpha": float(final_fit["alpha"]),
        "AIC": float(final_fit["aic"]),
        "EDF": float(final_fit.get("edf", np.nan)),
        "Parameter_Count": parameter_count,
        "Z_Weight_Column": MODEL_Z_WEIGHT_COL,
        "Z_Weight_Floor": z_weight_floor,
        "Age_Z_Violations": int(final_score[0]),
        "Max_Age_Z": float(final_score[1]),
    }
    search_table = pd.DataFrame(rows).sort_values(["Z_Violations", "Max_Z", "Excess_Z_Penalty", "AIC"]).reset_index(drop=True)
    if verbose:
        print(f"Selected knots={best_knots}  n_knots={best_n_knots}  z_viol={final_score[0]}  max_z={final_score[1]:.3f}")
    return {"knots": list(best_knots), "log": search_table, "fit": final_fit, "summary": summary}

def model_metrics(model_name, family_name, train_frame, holdout_frame, pred_train, pred_holdout, fit=None, formula=None, extra=None):
    actual_train = train_frame[MODEL_ACTUAL_COL].to_numpy(dtype=float)
    actual_holdout = holdout_frame[MODEL_ACTUAL_COL].to_numpy(dtype=float)
    expected_train = train_frame[MODEL_EXPECTED_COL].to_numpy(dtype=float)
    expected_holdout = holdout_frame[MODEL_EXPECTED_COL].to_numpy(dtype=float)
    pred_train = np.clip(np.asarray(pred_train, dtype=float), 1e-12, None)
    pred_holdout = np.clip(np.asarray(pred_holdout, dtype=float), 1e-12, None)

    metrics = pd.DataFrame({
        "Dataset": ["Train", "Holdout"],
        MODEL_ACTUAL_SUMMARY_COL: [actual_train.sum(), actual_holdout.sum()],
        MODEL_EXPECTED_SUMMARY_COL: [expected_train.sum(), expected_holdout.sum()],
        MODEL_PREDICTED_SUMMARY_COL: [pred_train.sum(), pred_holdout.sum()],
        "Actual_to_Predicted": [actual_train.sum() / pred_train.sum(), actual_holdout.sum() / pred_holdout.sum()],
        "Actual_to_VBT": [actual_train.sum() / expected_train.sum(), actual_holdout.sum() / expected_holdout.sum()],
        "Mean_Poisson_Deviance": [mean_poisson_deviance(actual_train, pred_train), mean_poisson_deviance(actual_holdout, pred_holdout)],
    })

    judging = {"Model": model_name, "Family": family_name, "AIC": np.nan, "BIC": np.nan, "Deviance": np.nan, "Null_Deviance": np.nan, "Parameters": np.nan}
    p_tables = []

    if isinstance(fit, dict):
        aics = []
        params = 0
        for label, item in fit.items():
            one_fit = item["fit"] if isinstance(item, dict) and "fit" in item else item
            aics.append(float(getattr(one_fit, "aic", np.nan)))
            params += len(getattr(one_fit, "params", []))
            if hasattr(one_fit, "pvalues"):
                p = pd.Series(one_fit.pvalues)
                b = pd.Series(one_fit.params).reindex(p.index)
                p_tables.append(pd.DataFrame({"Part": label, "Term": p.index, "Estimate": b.values, "P_Value": p.values}))
        judging["AIC"] = np.nansum(aics)
        judging["Parameters"] = params
    elif hasattr(fit, "statistics_"):
        stats = fit.statistics_
        judging["AIC"] = stats.get("AIC", np.nan)
        judging["GCV"] = stats.get("GCV", np.nan)
        judging["Effective_DF"] = stats.get("edof", np.nan)
        judging["Pseudo_R2"] = stats.get("pseudo_r2", {}).get("explained_deviance", np.nan)
        if "p_values" in stats:
            p_tables.append(pd.DataFrame({"Term": ["pyGAM term " + str(i) for i in range(len(stats["p_values"]))], "P_Value": stats["p_values"]}))
    elif fit is not None:
        judging["AIC"] = getattr(fit, "aic", np.nan)
        judging["BIC"] = getattr(fit, "bic_llf", getattr(fit, "bic", np.nan))
        judging["Deviance"] = getattr(fit, "deviance", np.nan)
        judging["Null_Deviance"] = getattr(fit, "null_deviance", np.nan)
        judging["Parameters"] = len(getattr(fit, "params", []))
        if hasattr(fit, "pvalues"):
            p = pd.Series(fit.pvalues)
            b = pd.Series(fit.params).reindex(p.index)
            p_tables.append(pd.DataFrame({"Term": p.index, "Estimate": b.values, "P_Value": p.values}))

    p_values = pd.concat(p_tables, ignore_index=True) if len(p_tables) else pd.DataFrame()
    if len(p_values):
        p_values["Significant_5pct"] = p_values["P_Value"] < 0.05
        judging["Smallest_P"] = p_values["P_Value"].min()
        judging["Terms_P_lt_0_05"] = int(p_values["Significant_5pct"].sum())
    if extra is not None:
        judging.update(extra)

    judging_metrics = pd.DataFrame([judging])
    model_results.append({
        **judging,
        "Train_A/P": metrics.loc[0, "Actual_to_Predicted"],
        "Holdout_A/P": metrics.loc[1, "Actual_to_Predicted"],
        "Train_Deviance": metrics.loc[0, "Mean_Poisson_Deviance"],
        "Holdout_Deviance": metrics.loc[1, "Mean_Poisson_Deviance"],
    })

    print("Model:", model_name)
    if formula is not None:
        print("Formula:", formula)
    if hasattr(fit, "summary"):
        print(fit.summary())
    if isinstance(fit, dict):
        for label, item in fit.items():
            one_fit = item["fit"] if isinstance(item, dict) and "fit" in item else item
            if hasattr(one_fit, "summary"):
                print("\nPart:", label)
                print(one_fit.summary())
    display(metrics)
    display(judging_metrics)
    return metrics, judging_metrics

def screen_feature_parent(encoded_feature: str) -> str:
    for feature in SCREEN_FEATURE_PARENT_ORDER:
        if encoded_feature == feature or encoded_feature.startswith(f"{feature}_"):
            return feature
    return encoded_feature


def summarize_screen_feature_importance(columns, importances, top_n: int = 40) -> tuple[pd.DataFrame, pd.DataFrame]:
    encoded = pd.DataFrame({"Encoded_Feature": list(columns), "Importance": importances})
    encoded["Feature"] = encoded["Encoded_Feature"].map(screen_feature_parent)
    encoded = encoded.sort_values("Importance", ascending=False).reset_index(drop=True)
    summary = (
        encoded.groupby("Feature", sort=False)
        .agg(
            Importance=("Importance", "sum"),
            Top_Encoded_Feature=("Encoded_Feature", "first"),
            Encoded_Feature_Count=("Encoded_Feature", "size"),
        )
        .sort_values("Importance", ascending=False)
        .head(top_n)
        .reset_index()
    )
    return summary, encoded


In [10]:
# Cell 7: GBM feature selection
screen_X = pd.get_dummies(screen_data[SCREEN_FEATURES], drop_first=False)
screen_y = np.log((screen_data[MODEL_ACTUAL_COL] + MODEL_LOG_SMOOTHING) / screen_data[MODEL_EXPOSURE_COL].clip(1e-10))
gbm_screen = GradientBoostingRegressor(random_state=42)
gbm_screen.fit(screen_X, screen_y, sample_weight=screen_data[MODEL_EXPOSURE_COL])
gbm_feature_selection, gbm_encoded_feature_importance = summarize_screen_feature_importance(
    screen_X.columns,
    gbm_screen.feature_importances_,
)
display(gbm_feature_selection)




,Feature,Importance,Top_Encoded_Feature,Encoded_Feature_Count
0,Face_Band_Group,0.493718,Face_Band_Group_01-04: <100k,2
1,Duration,0.297864,Duration,1
2,Attained_Age_mod,0.073159,Attained_Age_mod,1
3,Sex_x_Attained_Age_mod,0.067927,Sex_x_Attained_Age_mod,1
4,Issue_Age_mod,0.031790,Issue_Age_mod,1
5,Age_Ind,0.019212,Age_Ind_ALB,2
6,Observation_Year,0.007803,Observation_Year,1
7,Insurance_Plan_Group,0.004496,Insurance_Plan_Group_Term,2
8,Sex,0.004032,Sex_F,2


In [8]:
# Cell 8: ExtraTrees feature importance
screen_X = pd.get_dummies(screen_data[SCREEN_FEATURES], drop_first=False)
screen_y = np.log((screen_data[MODEL_ACTUAL_COL] + MODEL_LOG_SMOOTHING) / screen_data[MODEL_EXPOSURE_COL].clip(1e-10))
extra_screen = ExtraTreesRegressor(random_state=42, n_jobs=-1, min_samples_leaf=10)
extra_screen.fit(screen_X, screen_y, sample_weight=screen_data[MODEL_EXPOSURE_COL])
extra_feature_importance, extra_encoded_feature_importance = summarize_screen_feature_importance(
    screen_X.columns,
    extra_screen.feature_importances_,
)
display(extra_feature_importance)


,Feature,Importance,Top_Encoded_Feature,Encoded_Feature_Count
0,Face_Band_Group,0.463889,Face_Band_Group_05-11: >=100k,2
1,Duration,0.189564,Duration,1
2,Attained_Age_mod,0.120455,Attained_Age_mod,1
3,Sex_x_Attained_Age_mod,0.078236,Sex_x_Attained_Age_mod,1
4,Issue_Age_mod,0.053495,Issue_Age_mod,1
5,Observation_Year,0.038617,Observation_Year,1
6,Age_Ind,0.025499,Age_Ind_ALB,2
7,Sex,0.022463,Sex_M,2
8,Insurance_Plan_Group,0.007781,Insurance_Plan_Group_Perm,2


# Session 2 Model fitting candidates

In [9]:
# Cell 9: optional train/test split
TRAIN_FRACTION = 0.80
rng = np.random.default_rng(42)
values = df[[MODEL_EXPOSURE_COL, MODEL_ACTUAL_COL]].to_numpy(dtype=float)
totals = values.sum(axis=0)
target = totals[0] * TRAIN_FRACTION
best_score = np.inf
best_positions = np.arange(int(round(len(df) * TRAIN_FRACTION)))

for _ in range(4):
    order = rng.permutation(len(df))
    cut = int(np.searchsorted(np.cumsum(values[order, 0]), target, side="left"))
    for candidate_cut in sorted({max(1, min(len(df) - 1, cut)), max(1, min(len(df) - 1, cut + 1))}):
        positions = order[:candidate_cut]
        shares = values[positions].sum(axis=0) / totals
        score = np.max(np.abs(shares - TRAIN_FRACTION))
        if score < best_score:
            best_score = score
            best_positions = positions

train_mask = np.zeros(len(df), dtype=bool)
train_mask[best_positions] = True
train_raw = df.iloc[train_mask].copy().reset_index(drop=True)
holdout_raw = df.iloc[~train_mask].copy().reset_index(drop=True)
train_data = train_raw.groupby(MODEL_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
holdout_data = holdout_raw.groupby(MODEL_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
train_data["Attained_Age_mod_spline"] = train_data[ATTAINED_AGE_MOD_COL].clip(np.nextafter(ATTAINED_AGE_MIN, np.inf), np.nextafter(ATTAINED_AGE_MOD_MAX, -np.inf))
holdout_data["Attained_Age_mod_spline"] = holdout_data[ATTAINED_AGE_MOD_COL].clip(np.nextafter(ATTAINED_AGE_MIN, np.inf), np.nextafter(ATTAINED_AGE_MOD_MAX, -np.inf))
train_data["Attained_Age_Band"] = pd.cut(train_data[ATTAINED_AGE_MOD_COL], list(range(0, 75, 5)), right=False, include_lowest=True).astype(str)
holdout_data["Attained_Age_Band"] = pd.cut(holdout_data[ATTAINED_AGE_MOD_COL], list(range(0, 75, 5)), right=False, include_lowest=True).astype(str)

split_summary = pd.DataFrame({
    "Rows": [len(train_raw), len(holdout_raw), len(df)],
    MODEL_EXPOSURE_COL: [train_raw[MODEL_EXPOSURE_COL].sum(), holdout_raw[MODEL_EXPOSURE_COL].sum(), df[MODEL_EXPOSURE_COL].sum()],
    MODEL_ACTUAL_COL: [train_raw[MODEL_ACTUAL_COL].sum(), holdout_raw[MODEL_ACTUAL_COL].sum(), df[MODEL_ACTUAL_COL].sum()],
}, index=["Train", "Holdout", "Full scope"])
split_summary["Exposure_Share"] = split_summary[MODEL_EXPOSURE_COL] / split_summary.loc["Full scope", MODEL_EXPOSURE_COL]
split_summary["Target_Share"] = split_summary[MODEL_ACTUAL_COL] / split_summary.loc["Full scope", MODEL_ACTUAL_COL]
display(split_summary)




,Rows,Amount_Exposed,Death_Claim_Amount,Exposure_Share,Target_Share
Train,1139005,"2,231,771,099,799.459961",1546440491,0.800000,0.801043
Holdout,285609,"557,943,909,703.311523",384092474,0.200000,0.198957
Full scope,1424614,"2,789,715,009,502.771484",1930532965,1.000000,1.000000


In [12]:
# Cell 10: GLM for sex, attained age mod, and their interaction
glm_train = train_data.groupby(MODEL_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
glm_formula = f"{MODEL_ACTUAL_COL} ~ Sex_M + Attained_Age_mod + Sex_x_Attained_Age_mod"
glm_model = smf.glm(
    formula=glm_formula,
    data=glm_train,
    family=sm.families.Poisson(),
    offset=np.log(glm_train[MODEL_EXPOSURE_COL].clip(1e-10)),
).fit()

# Model quick judging metrics
pred_train = glm_model.predict(train_data, offset=np.log(train_data[MODEL_EXPOSURE_COL].clip(1e-10)))
pred_holdout = glm_model.predict(holdout_data, offset=np.log(holdout_data[MODEL_EXPOSURE_COL].clip(1e-10)))
metrics, judging_metrics = model_metrics("poisson_glm_sex_attained_age_mod_interaction", "Poisson", train_data, holdout_data, pred_train, pred_holdout, fit=glm_model, formula=glm_formula)
last_model = {"name": "poisson_glm_sex_attained_age_mod_interaction", "kind": "statsmodels_formula", "family": "Poisson", "formula": glm_formula}



Model: poisson_glm_sex_attained_age_mod_interaction
Formula: Death_Claim_Amount ~ Sex_M + Attained_Age_mod + Sex_x_Attained_Age_mod
                 Generalized Linear Model Regression Results                  
Dep. Variable:     Death_Claim_Amount   No. Observations:                  134
Model:                            GLM   Df Residuals:                      130
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1005e+08
Date:                Thu, 28 May 2026   Deviance:                   2.2010e+08
Time:                        06:03:00   Pearson chi2:                 1.99e+08
No. Iterations:                    35   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025

,Dataset,Actual_Death_Amount,Expected_VBT_Amount,Predicted_Death_Amount,Actual_to_Predicted,Actual_to_VBT,Mean_Poisson_Deviance
0,Train,"1,930,532,965.000000","1,625,002,296.821769","1,930,532,964.999995",1.000000,1.188019,"1,642,573.435735"
1,Holdout,"1,930,532,965.000000","1,625,002,296.821769","1,930,532,964.999995",1.000000,1.188019,"1,642,573.435735"


,Model,Family,AIC,BIC,Deviance,Null_Deviance,Parameters,Smallest_P,Terms_P_lt_0_05
0,poisson_glm_sex_attained_age_mod_interaction,Poisson,"220,107,255.576469","220,107,267.167829","220,104,840.388432","3,508,196,128.359365",4,0.000000,4


In [ ]:
# Cell 11: Poisson GAM, same predictors, with manual knot input
MANUAL_KNOTS = [10, 25, 45]
gam_train = train_data.groupby(MODEL_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
gam_spline = f"bs(Attained_Age_mod, knots={tuple(MANUAL_KNOTS)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
gam_formula = f"{MODEL_ACTUAL_COL} ~ Sex_M + " + gam_spline + " + Sex_M:" + gam_spline
gam_manual = smf.glm(
    formula=gam_formula,
    data=gam_train,
    family=sm.families.Poisson(),
    offset=np.log(gam_train[MODEL_EXPOSURE_COL].clip(1e-10)),
).fit()

# Model quick judging metrics
pred_train = gam_manual.predict(train_data, offset=np.log(train_data[MODEL_EXPOSURE_COL].clip(1e-10)))
pred_holdout = gam_manual.predict(holdout_data, offset=np.log(holdout_data[MODEL_EXPOSURE_COL].clip(1e-10)))
metrics, judging_metrics = model_metrics("poisson_gam_manual_knots", "Poisson", train_data, holdout_data, pred_train, pred_holdout, fit=gam_manual, formula=gam_formula, extra={"Knots": str(MANUAL_KNOTS)})
last_model = {"name": "poisson_gam_manual_knots", "kind": "statsmodels_formula", "family": "Poisson", "formula": gam_formula, "knots": MANUAL_KNOTS}




In [ ]:
# Cell 12: Poisson GAM, same predictors, knot search, with penalty term
KNOT_MIN_KNOTS = 1
KNOT_MAX_KNOTS = 5
KNOT_MIN_SEP = 8
KNOT_BOUNDARY_GAP = 6
KNOT_CANDIDATES = sorted(list(range(1, 35, 1)) + list(range(35, 67, 5)))

knot_search = search_knots(
    train_data,
    family=sm.families.Poisson(),
    min_knots=KNOT_MIN_KNOTS,
    max_knots=KNOT_MAX_KNOTS,
    min_sep=KNOT_MIN_SEP,
    boundary_gap=KNOT_BOUNDARY_GAP,
    raw_candidates=KNOT_CANDIDATES,
)
search_knots_found = knot_search["knots"]
knot_search_table = knot_search["log"]
knot_search_summary = pd.DataFrame([knot_search["summary"]])
gam_search = knot_search["fit"]

# Model quick judging metrics
pred_train = gam_search["wrapper"].predict(train_data, offset=np.log(train_data[MODEL_EXPOSURE_COL].clip(1e-10)).values)
pred_holdout = gam_search["wrapper"].predict(holdout_data, offset=np.log(holdout_data[MODEL_EXPOSURE_COL].clip(1e-10)).values)
print("Chosen knots:", search_knots_found)
display(knot_search_summary)
metrics, judging_metrics = model_metrics(
    "poisson_gam_knot_search", "Poisson penalized GAM",
    train_data, holdout_data, pred_train, pred_holdout,
    fit=None,
    formula="penalized combined-sex spline on Sex, Attained_Age_mod, and their interaction",
    extra={**knot_search["summary"], "Knots": str(search_knots_found)},
)
last_model = {
    "name": "poisson_gam_knot_search",
    "kind": "penalized_combined_gam",
    "family": "Poisson",
    "knots": search_knots_found,
    "alpha": knot_search["summary"]["Alpha"],
    "min_sep": KNOT_MIN_SEP,
    "boundary_gap": KNOT_BOUNDARY_GAP,
}


In [ ]:
# Cell 13: Poisson GAM, same predictors, auto knot
auto_knots = sorted(train_data["Attained_Age_mod"].quantile([0.25, 0.50, 0.75]).round(1).unique().tolist())
gam_train = train_data.groupby(MODEL_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
gam_spline = f"bs(Attained_Age_mod, knots={tuple(auto_knots)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
gam_formula = f"{MODEL_ACTUAL_COL} ~ Sex_M + " + gam_spline + " + Sex_M:" + gam_spline
gam_auto = smf.glm(
    formula=gam_formula,
    data=gam_train,
    family=sm.families.Poisson(),
    offset=np.log(gam_train[MODEL_EXPOSURE_COL].clip(1e-10)),
).fit()

# Model quick judging metrics
pred_train = gam_auto.predict(train_data, offset=np.log(train_data[MODEL_EXPOSURE_COL].clip(1e-10)))
pred_holdout = gam_auto.predict(holdout_data, offset=np.log(holdout_data[MODEL_EXPOSURE_COL].clip(1e-10)))
metrics, judging_metrics = model_metrics("poisson_gam_auto_knots", "Poisson", train_data, holdout_data, pred_train, pred_holdout, fit=gam_auto, formula=gam_formula, extra={"Knots": str(auto_knots)})
last_model = {"name": "poisson_gam_auto_knots", "kind": "statsmodels_formula", "family": "Poisson", "formula": gam_formula, "knots": auto_knots}




In [ ]:
# Cell 14: NegativeBinomial GAM, same predictors, auto knot
nb_knots = sorted(train_data["Attained_Age_mod"].quantile([0.25, 0.50, 0.75]).round(1).unique().tolist())
gam_train = train_data.groupby(MODEL_FEATURES, dropna=False)[METRIC_COLS].sum().reset_index()
gam_spline = f"bs(Attained_Age_mod, knots={tuple(nb_knots)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
gam_formula = f"{MODEL_ACTUAL_COL} ~ Sex_M + " + gam_spline + " + Sex_M:" + gam_spline
nb_gam = smf.glm(
    formula=gam_formula,
    data=gam_train,
    family=sm.families.NegativeBinomial(alpha=1.0),
    offset=np.log(gam_train[MODEL_EXPOSURE_COL].clip(1e-10)),
).fit()

# Model quick judging metrics
pred_train = nb_gam.predict(train_data, offset=np.log(train_data[MODEL_EXPOSURE_COL].clip(1e-10)))
pred_holdout = nb_gam.predict(holdout_data, offset=np.log(holdout_data[MODEL_EXPOSURE_COL].clip(1e-10)))
metrics, judging_metrics = model_metrics("negative_binomial_gam_auto_knots", "NegativeBinomial", train_data, holdout_data, pred_train, pred_holdout, fit=nb_gam, formula=gam_formula, extra={"Knots": str(nb_knots)})
last_model = {"name": "negative_binomial_gam_auto_knots", "kind": "statsmodels_formula", "family": "NegativeBinomial", "formula": gam_formula, "knots": nb_knots}


In [ ]:
# Cell 15: Poisson GAM per-sex knot search
PER_SEX_MIN_SEP = {"F": 7, "M": 7}
PER_SEX_BOUNDARY_GAP = {"F": 4, "M": 8}
PER_SEX_MAX_KNOTS = {"F": 3, "M": 5}
PER_SEX_MIN_KNOTS = 1
PER_SEX_CANDIDATES = sorted(list(range(1, 35, 2)) + list(range(35, 67, 5)))

per_sex_models = {}
per_sex_knots = {}
per_sex_tables = []
pred_train = np.zeros(len(train_data))
pred_holdout = np.zeros(len(holdout_data))
for sex_value in sorted(train_data["Sex"].astype(str).unique()):
    knot_result = search_knots(
        train_data,
        sex_value=sex_value,
        family=sm.families.Poisson(),
        min_knots=PER_SEX_MIN_KNOTS,
        max_knots=PER_SEX_MAX_KNOTS.get(sex_value, 3),
        min_sep=PER_SEX_MIN_SEP.get(sex_value, 7),
        boundary_gap=PER_SEX_BOUNDARY_GAP.get(sex_value, 6),
        raw_candidates=PER_SEX_CANDIDATES,
    )
    knots = knot_result["knots"]
    table = knot_result["log"]
    per_sex_knots[sex_value] = knots
    table.insert(0, "Sex", sex_value)
    per_sex_tables.append(table.head(10))
    sex_train = train_data[train_data["Sex"].astype(str) == sex_value].groupby(["Attained_Age_mod"], dropna=False)[METRIC_COLS].sum().reset_index()
    formula = f"{MODEL_ACTUAL_COL} ~ bs(Attained_Age_mod, knots={tuple(knots)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
    fit = smf.glm(formula=formula, data=sex_train, family=sm.families.Poisson(), offset=np.log(sex_train[MODEL_EXPOSURE_COL].clip(1e-10))).fit()
    per_sex_models[sex_value] = {"fit": fit, "formula": formula}
    train_mask = train_data["Sex"].astype(str).to_numpy() == sex_value
    holdout_mask = holdout_data["Sex"].astype(str).to_numpy() == sex_value
    pred_train[train_mask] = fit.predict(train_data.loc[train_mask], offset=np.log(train_data.loc[train_mask, MODEL_EXPOSURE_COL].clip(1e-10)))
    pred_holdout[holdout_mask] = fit.predict(holdout_data.loc[holdout_mask], offset=np.log(holdout_data.loc[holdout_mask, MODEL_EXPOSURE_COL].clip(1e-10)))

# Model quick judging metrics
print("Chosen knots by sex:", per_sex_knots)
metrics, judging_metrics = model_metrics("poisson_gam_per_sex_knot_search", "Poisson", train_data, holdout_data, pred_train, pred_holdout, fit=per_sex_models, formula="separate Poisson spline by sex", extra={"Knots": str(per_sex_knots)})
last_model = {"name": "poisson_gam_per_sex_knot_search", "kind": "per_sex_formula", "family": "Poisson", "knots": per_sex_knots}


In [ ]:
# Cell 16: Poisson GAM per-sex auto knot
per_sex_models = {}
per_sex_knots = {}
pred_train = np.zeros(len(train_data))
pred_holdout = np.zeros(len(holdout_data))

for sex_value in sorted(train_data["Sex"].astype(str).unique()):
    sex_rows = train_data[train_data["Sex"].astype(str) == sex_value]
    knots = sorted(sex_rows["Attained_Age_mod"].quantile([0.25, 0.50, 0.75]).round(1).unique().tolist())
    per_sex_knots[sex_value] = knots
    sex_train = sex_rows.groupby(["Attained_Age_mod"], dropna=False)[METRIC_COLS].sum().reset_index()
    formula = f"{MODEL_ACTUAL_COL} ~ bs(Attained_Age_mod, knots={tuple(knots)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
    fit = smf.glm(formula=formula, data=sex_train, family=sm.families.Poisson(), offset=np.log(sex_train[MODEL_EXPOSURE_COL].clip(1e-10))).fit()
    per_sex_models[sex_value] = {"fit": fit, "formula": formula}
    train_mask = train_data["Sex"].astype(str).to_numpy() == sex_value
    holdout_mask = holdout_data["Sex"].astype(str).to_numpy() == sex_value
    pred_train[train_mask] = fit.predict(train_data.loc[train_mask], offset=np.log(train_data.loc[train_mask, MODEL_EXPOSURE_COL].clip(1e-10)))
    pred_holdout[holdout_mask] = fit.predict(holdout_data.loc[holdout_mask], offset=np.log(holdout_data.loc[holdout_mask, MODEL_EXPOSURE_COL].clip(1e-10)))

# Model quick judging metrics
print("Auto knots by sex:", per_sex_knots)
metrics, judging_metrics = model_metrics("poisson_gam_per_sex_auto_knots", "Poisson", train_data, holdout_data, pred_train, pred_holdout, fit=per_sex_models, formula="separate Poisson spline by sex", extra={"Knots": str(per_sex_knots)})
last_model = {"name": "poisson_gam_per_sex_auto_knots", "kind": "per_sex_formula", "family": "Poisson", "knots": per_sex_knots}




In [ ]:
# Cell 17: Whittaker-Henderson, auto
WH_LAMBDA = round(len(train_data) / 1000, 1)
wh_tables = {}

for sex_value in sorted(train_data["Sex"].astype(str).unique()):
    wh = train_data[train_data["Sex"].astype(str) == sex_value].groupby("Attained_Age_mod", dropna=False)[[MODEL_ACTUAL_COL, MODEL_EXPOSURE_COL]].sum().reset_index().sort_values("Attained_Age_mod")
    age = wh["Attained_Age_mod"].to_numpy(dtype=float)
    exposure = wh[MODEL_EXPOSURE_COL].to_numpy(dtype=float).clip(1e-10)
    crude_rate = wh[MODEL_ACTUAL_COL].to_numpy(dtype=float) / exposure
    D = np.eye(len(wh))
    for _ in range(2):
        D = np.diff(D, axis=0)
    W = np.diag(exposure)
    rate = np.linalg.solve(W + WH_LAMBDA * (D.T @ D), W @ crude_rate)
    wh_tables[sex_value] = (age, np.clip(rate, 1e-12, None))

# Model quick judging metrics
pred_train = np.zeros(len(train_data))
pred_holdout = np.zeros(len(holdout_data))
for sex_value, (age, rate) in wh_tables.items():
    train_mask = train_data["Sex"].astype(str).to_numpy() == sex_value
    holdout_mask = holdout_data["Sex"].astype(str).to_numpy() == sex_value
    pred_train[train_mask] = np.interp(train_data.loc[train_mask, "Attained_Age_mod"], age, rate) * train_data.loc[train_mask, MODEL_EXPOSURE_COL]
    pred_holdout[holdout_mask] = np.interp(holdout_data.loc[holdout_mask, "Attained_Age_mod"], age, rate) * holdout_data.loc[holdout_mask, MODEL_EXPOSURE_COL]

display(pd.DataFrame([{"Sex": k, "Age_Nodes": len(v[0]), "Min_Rate": v[1].min(), "Max_Rate": v[1].max()} for k, v in wh_tables.items()]))
metrics, judging_metrics = model_metrics("whittaker_henderson_auto", "WH", train_data, holdout_data, pred_train, pred_holdout, formula="per-sex WH graduation on attained age; second differences penalized", extra={"Lambda": WH_LAMBDA})
last_model = {"name": "whittaker_henderson_auto", "kind": "wh", "lambda": WH_LAMBDA}


In [ ]:
# Cell 18: PPR
ppr_X_train = train_data[ML_FEATURES].to_numpy(dtype=float)
ppr_X_holdout = holdout_data[ML_FEATURES].to_numpy(dtype=float)
y_train = train_data[MODEL_ACTUAL_COL].to_numpy(dtype=float)
E_train = train_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float).clip(1e-10)
E_holdout = holdout_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float).clip(1e-10)
ppr_model = PoissonGAM(gam_f(0) + s(1) + s(2)).fit(ppr_X_train, y_train, exposure=E_train)

# Model quick judging metrics
pred_train = ppr_model.predict(ppr_X_train, exposure=E_train)
pred_holdout = ppr_model.predict(ppr_X_holdout, exposure=E_holdout)
metrics, judging_metrics = model_metrics("ppr_poisson_gam", "PoissonGAM", train_data, holdout_data, pred_train, pred_holdout, fit=ppr_model, formula="pyGAM: f(Sex_M) + s(Attained_Age_mod) + s(Sex_x_Attained_Age_mod)")
last_model = {"name": "ppr_poisson_gam", "kind": "pygam"}




In [ ]:
# Cell 19: Poisson regressor
X_train = train_data[ML_FEATURES]
X_holdout = holdout_data[ML_FEATURES]
y_train = train_data[MODEL_ACTUAL_COL] / train_data[MODEL_EXPOSURE_COL].clip(1e-10)
poisson_regressor = PoissonRegressor(alpha=0.001, max_iter=500)
poisson_regressor.fit(X_train, y_train, sample_weight=train_data[MODEL_EXPOSURE_COL])

# Model quick judging metrics
pred_train = poisson_regressor.predict(X_train) * train_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
pred_holdout = poisson_regressor.predict(X_holdout) * holdout_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
metrics, judging_metrics = model_metrics("poisson_regressor", "Sklearn Poisson", train_data, holdout_data, pred_train, pred_holdout, fit=poisson_regressor, extra={"Estimator": str(poisson_regressor), "Features": X_train.shape[1]})
last_model = {"name": "poisson_regressor", "kind": "sklearn_rate", "model": poisson_regressor, "columns": X_train.columns.tolist()}


In [ ]:
# Cell 20: Random forest
X_train = train_data[ML_FEATURES]
X_holdout = holdout_data[ML_FEATURES]
y_train = np.log((train_data[MODEL_ACTUAL_COL] + MODEL_LOG_SMOOTHING) / train_data[MODEL_EXPOSURE_COL].clip(1e-10))
random_forest = RandomForestRegressor(random_state=42, n_jobs=-1, min_samples_leaf=10)
random_forest.fit(X_train, y_train, sample_weight=train_data[MODEL_EXPOSURE_COL])

# Model quick judging metrics
pred_train = np.exp(random_forest.predict(X_train)) * train_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
pred_holdout = np.exp(random_forest.predict(X_holdout)) * holdout_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
metrics, judging_metrics = model_metrics("random_forest_log_rate", "RandomForest", train_data, holdout_data, pred_train, pred_holdout, fit=random_forest, extra={"Estimator": str(random_forest), "Features": X_train.shape[1]})
last_model = {"name": "random_forest_log_rate", "kind": "sklearn_log_rate", "model": random_forest, "columns": X_train.columns.tolist()}


In [ ]:
# Cell 21: Gradient boosting
X_train = train_data[ML_FEATURES]
X_holdout = holdout_data[ML_FEATURES]
y_train = np.log((train_data[MODEL_ACTUAL_COL] + MODEL_LOG_SMOOTHING) / train_data[MODEL_EXPOSURE_COL].clip(1e-10))
gradient_boosting = HistGradientBoostingRegressor(random_state=42)
gradient_boosting.fit(X_train, y_train, sample_weight=train_data[MODEL_EXPOSURE_COL])

# Model quick judging metrics
pred_train = np.exp(gradient_boosting.predict(X_train)) * train_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
pred_holdout = np.exp(gradient_boosting.predict(X_holdout)) * holdout_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
metrics, judging_metrics = model_metrics("gradient_boosting_log_rate", "HistGradientBoosting", train_data, holdout_data, pred_train, pred_holdout, fit=gradient_boosting, extra={"Estimator": str(gradient_boosting), "Features": X_train.shape[1]})
last_model = {"name": "gradient_boosting_log_rate", "kind": "sklearn_log_rate", "model": gradient_boosting, "columns": X_train.columns.tolist()}


In [ ]:
# Cell 22: ExtraTrees
X_train = train_data[ML_FEATURES]
X_holdout = holdout_data[ML_FEATURES]
y_train = np.log((train_data[MODEL_ACTUAL_COL] + MODEL_LOG_SMOOTHING) / train_data[MODEL_EXPOSURE_COL].clip(1e-10))
extra_trees = ExtraTreesRegressor(random_state=42, n_jobs=-1, min_samples_leaf=10)
extra_trees.fit(X_train, y_train, sample_weight=train_data[MODEL_EXPOSURE_COL])

# Model quick judging metrics
pred_train = np.exp(extra_trees.predict(X_train)) * train_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
pred_holdout = np.exp(extra_trees.predict(X_holdout)) * holdout_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
metrics, judging_metrics = model_metrics("extra_trees_log_rate", "ExtraTrees", train_data, holdout_data, pred_train, pred_holdout, fit=extra_trees, extra={"Estimator": str(extra_trees), "Features": X_train.shape[1]})
last_model = {"name": "extra_trees_log_rate", "kind": "sklearn_log_rate", "model": extra_trees, "columns": X_train.columns.tolist()}


# Session 3 Review and output

In [ ]:
# Cell 23: refit to full scope on last run model
print("Refitting last run model:", last_model["name"])
final_model = last_model.copy()

refit_features = ["Sex", "Sex_M", "Attained_Age_mod", "Sex_x_Attained_Age_mod"]
refit_ml_features = ["Sex_M", "Attained_Age_mod", "Sex_x_Attained_Age_mod"]
refit_data = model_data.copy()
refit_data["Sex_x_Attained_Age_mod"] = refit_data["Sex_M"] * refit_data["Attained_Age_mod"]

if last_model["kind"] == "penalized_combined_gam":
    knots = list(last_model["knots"])
    alpha = float(last_model["alpha"])
    refit_result = search_knots(
        refit_data,
        min_knots=len(knots),
        max_knots=len(knots),
        min_sep=last_model.get("min_sep", 8),
        boundary_gap=last_model.get("boundary_gap", 6),
        raw_candidates=knots,
        search_alphas=[alpha],
        final_alphas=[alpha],
        n_workers=1,
        verbose=False,
    )
    final_fit = refit_result["fit"]
    final_pred = final_fit["wrapper"].predict(refit_data, offset=np.log(refit_data[MODEL_EXPOSURE_COL].clip(1e-10)).values)
    final_model["fit"] = final_fit
    final_model["fit_stats"] = refit_result["summary"]
elif last_model["kind"] == "statsmodels_formula":
    family = sm.families.Poisson() if last_model["family"] == "Poisson" else sm.families.NegativeBinomial(alpha=1.0)
    refit_formula = re.sub(r"Sex_x_Attained_Age(?!_mod)", "Sex_x_Attained_Age_mod", last_model["formula"])
    refit_formula = refit_formula.replace("Attained_Age_mod_spline", "Attained_Age_mod")
    fit_data = refit_data.groupby(refit_features, dropna=False)[METRIC_COLS].sum().reset_index()
    final_fit = smf.glm(refit_formula, data=fit_data, family=family, offset=np.log(fit_data[MODEL_EXPOSURE_COL].clip(1e-10))).fit()
    final_pred = final_fit.predict(refit_data, offset=np.log(refit_data[MODEL_EXPOSURE_COL].clip(1e-10)))
    final_model["formula"] = refit_formula
    final_model["fit"] = final_fit
elif last_model["kind"] == "per_sex_formula":
    final_fit = {}
    final_pred = np.zeros(len(refit_data))
    for sex_value, knots in last_model["knots"].items():
        sex_data = refit_data[refit_data["Sex"].astype(str) == sex_value].groupby("Attained_Age_mod", dropna=False)[METRIC_COLS].sum().reset_index()
        formula = f"{MODEL_ACTUAL_COL} ~ bs(Attained_Age_mod, knots={tuple(knots)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
        fit = smf.glm(formula, data=sex_data, family=sm.families.Poisson(), offset=np.log(sex_data[MODEL_EXPOSURE_COL].clip(1e-10))).fit()
        mask = refit_data["Sex"].astype(str).to_numpy() == sex_value
        final_pred[mask] = fit.predict(refit_data.loc[mask], offset=np.log(refit_data.loc[mask, MODEL_EXPOSURE_COL].clip(1e-10)))
        final_fit[sex_value] = fit
    final_model["fit"] = final_fit
elif last_model["kind"] == "wh":
    final_fit = {}
    final_pred = np.zeros(len(refit_data))
    for sex_value in sorted(refit_data["Sex"].astype(str).unique()):
        wh = refit_data[refit_data["Sex"].astype(str) == sex_value].groupby("Attained_Age_mod", dropna=False)[[MODEL_ACTUAL_COL, MODEL_EXPOSURE_COL]].sum().reset_index().sort_values("Attained_Age_mod")
        age = wh["Attained_Age_mod"].to_numpy(dtype=float)
        exposure = wh[MODEL_EXPOSURE_COL].to_numpy(dtype=float).clip(1e-10)
        crude = wh[MODEL_ACTUAL_COL].to_numpy(dtype=float) / exposure
        D = np.eye(len(wh))
        for _ in range(2):
            D = np.diff(D, axis=0)
        rate = np.linalg.solve(np.diag(exposure) + last_model["lambda"] * (D.T @ D), np.diag(exposure) @ crude)
        mask = refit_data["Sex"].astype(str).to_numpy() == sex_value
        final_pred[mask] = np.interp(refit_data.loc[mask, "Attained_Age_mod"], age, rate) * refit_data.loc[mask, MODEL_EXPOSURE_COL]
        final_fit[sex_value] = (age, rate)
    final_model["fit"] = final_fit
elif last_model["kind"] == "pygam":
    X_full = refit_data[refit_ml_features].to_numpy(dtype=float)
    final_fit = PoissonGAM(gam_f(0) + s(1) + s(2)).fit(X_full, refit_data[MODEL_ACTUAL_COL], exposure=refit_data[MODEL_EXPOSURE_COL].clip(1e-10))
    final_pred = final_fit.predict(X_full, exposure=refit_data[MODEL_EXPOSURE_COL].clip(1e-10))
    final_model["fit"] = final_fit
    final_model["columns"] = refit_ml_features
elif last_model["kind"] == "sklearn_rate":
    X_full = refit_data[refit_ml_features]
    final_fit = clone(last_model["model"]).fit(X_full, refit_data[MODEL_ACTUAL_COL] / refit_data[MODEL_EXPOSURE_COL].clip(1e-10), sample_weight=refit_data[MODEL_EXPOSURE_COL])
    final_pred = final_fit.predict(X_full) * refit_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
    final_model["fit"] = final_fit
    final_model["columns"] = X_full.columns.tolist()
elif last_model["kind"] == "sklearn_log_rate":
    X_full = refit_data[refit_ml_features]
    final_fit = clone(last_model["model"]).fit(X_full, np.log((refit_data[MODEL_ACTUAL_COL] + MODEL_LOG_SMOOTHING) / refit_data[MODEL_EXPOSURE_COL].clip(1e-10)), sample_weight=refit_data[MODEL_EXPOSURE_COL])
    final_pred = np.exp(final_fit.predict(X_full)) * refit_data[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
    final_model["fit"] = final_fit
    final_model["columns"] = X_full.columns.tolist()

scored_df = refit_data.copy()
scored_df[MODEL_PREDICTED_COL] = np.clip(final_pred, 1e-12, None)
scored_df["Actual_to_Predicted"] = scored_df[MODEL_ACTUAL_COL] / scored_df[MODEL_PREDICTED_COL]
scored_df["Actual_to_VBT"] = scored_df[MODEL_ACTUAL_COL] / scored_df[MODEL_EXPECTED_COL]
scored_df["Actual_Rate_per_1000"] = scored_df[MODEL_ACTUAL_COL] * 1000 / scored_df[MODEL_EXPOSURE_COL]
scored_df["Expected_Rate_per_1000"] = scored_df[MODEL_EXPECTED_COL] * 1000 / scored_df[MODEL_EXPOSURE_COL]
scored_df["Predicted_Rate_per_1000"] = scored_df[MODEL_PREDICTED_COL] * 1000 / scored_df[MODEL_EXPOSURE_COL]

final_metrics = pd.DataFrame({
    "Model": [final_model["name"]],
    "Rows": [len(scored_df)],
    MODEL_ACTUAL_SUMMARY_COL: [scored_df[MODEL_ACTUAL_COL].sum()],
    MODEL_EXPECTED_SUMMARY_COL: [scored_df[MODEL_EXPECTED_COL].sum()],
    MODEL_PREDICTED_SUMMARY_COL: [scored_df[MODEL_PREDICTED_COL].sum()],
    "Actual_to_Predicted": [scored_df[MODEL_ACTUAL_COL].sum() / scored_df[MODEL_PREDICTED_COL].sum()],
    "Actual_to_VBT": [scored_df[MODEL_ACTUAL_COL].sum() / scored_df[MODEL_EXPECTED_COL].sum()],
    "Mean_Poisson_Deviance": [mean_poisson_deviance(scored_df[MODEL_ACTUAL_COL], scored_df[MODEL_PREDICTED_COL])],
})
display(final_metrics)
if "fit_stats" in final_model:
    display(pd.DataFrame([final_model["fit_stats"]]))




In [ ]:
# Cell 24: K-fold validation
K_FOLDS = 5
folded = scored_df.sample(frac=1, random_state=42).reset_index(drop=True)
folded["Fold"] = np.arange(len(folded)) % K_FOLDS
fold_rows = []

for fold in range(K_FOLDS):
    part = folded[folded["Fold"] == fold]
    fold_rows.append({
        "Fold": fold + 1,
        "Rows": len(part),
        MODEL_ACTUAL_SUMMARY_COL: part[MODEL_ACTUAL_COL].sum(),
        MODEL_PREDICTED_SUMMARY_COL: part[MODEL_PREDICTED_COL].sum(),
        "Actual_to_Predicted": part[MODEL_ACTUAL_COL].sum() / part[MODEL_PREDICTED_COL].sum(),
        "Actual_to_VBT": part[MODEL_ACTUAL_COL].sum() / part[MODEL_EXPECTED_COL].sum(),
        "Mean_Poisson_Deviance": mean_poisson_deviance(part[MODEL_ACTUAL_COL], part[MODEL_PREDICTED_COL]),
    })

kfold_validation = pd.DataFrame(fold_rows)
display(kfold_validation)




In [ ]:
# Cell 25: bootstrap validation
N_BOOTSTRAP = 200
rng = np.random.default_rng(42)
boot_rows = []

for i in range(N_BOOTSTRAP):
    part = scored_df.iloc[rng.integers(0, len(scored_df), len(scored_df))]
    boot_rows.append({
        "Sample": i + 1,
        "Actual_to_Predicted": part[MODEL_ACTUAL_COL].sum() / part[MODEL_PREDICTED_COL].sum(),
        "Actual_to_VBT": part[MODEL_ACTUAL_COL].sum() / part[MODEL_EXPECTED_COL].sum(),
        "Mean_Poisson_Deviance": mean_poisson_deviance(part[MODEL_ACTUAL_COL], part[MODEL_PREDICTED_COL]),
    })

bootstrap_validation = pd.DataFrame(boot_rows)
bootstrap_summary = bootstrap_validation.drop(columns="Sample").agg(["mean", "std", "min", "max"]).T
display(bootstrap_summary)




In [ ]:
# Cell 26: review tables output
review_tables = {}

for group_col in ["Sex", "Attained_Age_Band"]:
    table = scored_df.groupby(group_col, dropna=False)[[MODEL_ACTUAL_COL, MODEL_EXPECTED_COL, MODEL_PREDICTED_COL, MODEL_EXPOSURE_COL]].sum().reset_index()
    table["Actual_to_Predicted"] = table[MODEL_ACTUAL_COL] / table[MODEL_PREDICTED_COL]
    table["Actual_to_VBT"] = table[MODEL_ACTUAL_COL] / table[MODEL_EXPECTED_COL]
    table["Predicted_Rate_per_1000"] = table[MODEL_PREDICTED_COL] * 1000 / table[MODEL_EXPOSURE_COL]
    review_tables[group_col] = table
    print("\n", group_col)
    display(table)




In [ ]:
# Cell 27: Target-style graph review
MIN_Z_WEIGHT_FOR_MONOTONICITY = MIN_Z_WEIGHT_FOR_REVIEW

plot_scored_df = df.copy()
plot_scored_df["Sex_x_Attained_Age_mod"] = plot_scored_df["Sex_M"] * plot_scored_df["Attained_Age_mod"]

if final_model["kind"] == "penalized_combined_gam":
    plot_pred = final_model["fit"]["wrapper"].predict(
        plot_scored_df,
        offset=np.log(plot_scored_df[MODEL_EXPOSURE_COL].clip(1e-10)).values,
    )
elif final_model["kind"] == "statsmodels_formula":
    plot_pred = final_model["fit"].predict(
        plot_scored_df,
        offset=np.log(plot_scored_df[MODEL_EXPOSURE_COL].clip(1e-10)),
    )
elif final_model["kind"] == "per_sex_formula":
    plot_pred = np.zeros(len(plot_scored_df))
    for sex_value, fit in final_model["fit"].items():
        mask = plot_scored_df["Sex"].astype(str).to_numpy() == sex_value
        plot_pred[mask] = fit.predict(
            plot_scored_df.loc[mask],
            offset=np.log(plot_scored_df.loc[mask, MODEL_EXPOSURE_COL].clip(1e-10)),
        )
elif final_model["kind"] == "wh":
    plot_pred = np.zeros(len(plot_scored_df))
    for sex_value, fit in final_model["fit"].items():
        age, rate = fit
        mask = plot_scored_df["Sex"].astype(str).to_numpy() == sex_value
        plot_pred[mask] = np.interp(plot_scored_df.loc[mask, "Attained_Age_mod"], age, rate) * plot_scored_df.loc[mask, MODEL_EXPOSURE_COL]
elif final_model["kind"] == "pygam":
    x_plot = plot_scored_df[final_model.get("columns", ML_FEATURES)].to_numpy(dtype=float)
    plot_pred = final_model["fit"].predict(x_plot, exposure=plot_scored_df[MODEL_EXPOSURE_COL].clip(1e-10))
elif final_model["kind"] == "sklearn_rate":
    x_plot = plot_scored_df[final_model.get("columns", ML_FEATURES)]
    plot_pred = final_model["fit"].predict(x_plot) * plot_scored_df[MODEL_EXPOSURE_COL].to_numpy(dtype=float)
elif final_model["kind"] == "sklearn_log_rate":
    x_plot = plot_scored_df[final_model.get("columns", ML_FEATURES)]
    plot_pred = np.exp(final_model["fit"].predict(x_plot)) * plot_scored_df[MODEL_EXPOSURE_COL].to_numpy(dtype=float)

plot_scored_df[MODEL_PREDICTED_COL] = np.clip(plot_pred, 1e-12, None)
plot_scored_df["Actual_Rate_per_1000"] = plot_scored_df[MODEL_ACTUAL_COL] * 1000 / plot_scored_df[MODEL_EXPOSURE_COL]
plot_scored_df["Expected_Rate_per_1000"] = plot_scored_df[MODEL_EXPECTED_COL] * 1000 / plot_scored_df[MODEL_EXPOSURE_COL]
plot_scored_df["Predicted_Rate_per_1000"] = plot_scored_df[MODEL_PREDICTED_COL] * 1000 / plot_scored_df[MODEL_EXPOSURE_COL]
diagnostic_df = plot_scored_df.copy()

fig, axes = plt.subplots(3, 2, figsize=(18, 16), constrained_layout=True)
axes = axes.ravel()
plot_metric_cols = list(dict.fromkeys([MODEL_ACTUAL_COL, MODEL_PREDICTED_COL, MODEL_EXPECTED_COL, MODEL_EXPOSURE_COL, MODEL_Z_WEIGHT_COL]))

issue_plot = diagnostic_df.groupby(["Sex", "Issue_Age_mod"], dropna=False)[plot_metric_cols].sum().reset_index()
issue_plot = issue_plot[issue_plot[MODEL_Z_WEIGHT_COL] >= MIN_Z_WEIGHT_FOR_MONOTONICITY].copy()
issue_plot["Actual_Rate_per_1000"] = issue_plot[MODEL_ACTUAL_COL] * 1000 / issue_plot[MODEL_EXPOSURE_COL]
issue_plot["Predicted_Rate_per_1000"] = issue_plot[MODEL_PREDICTED_COL] * 1000 / issue_plot[MODEL_EXPOSURE_COL]
for sex_value, part in issue_plot.groupby("Sex", dropna=False):
    part = part.sort_values("Issue_Age_mod")
    axes[0].plot(part["Issue_Age_mod"], part["Actual_Rate_per_1000"], marker="o", linestyle="-", label=f"{sex_value} | Actual")
    axes[0].plot(part["Issue_Age_mod"], part["Predicted_Rate_per_1000"], marker="s", linestyle="--", label=f"{sex_value} | Predicted")
axes[0].set_title(f"Actual vs predicted {MODEL_TARGET_LABEL.lower()} rate by Issue_Age_mod split by Sex")
axes[0].set_xlabel("Issue_Age_mod")
axes[0].set_ylabel(f"Rate per 1000 {MODEL_RATE_DENOMINATOR_LABEL}")
axes[0].grid(True, alpha=0.3)
axes[0].legend(title="Sex", fontsize=8, title_fontsize=9)
axes[0].ticklabel_format(style="plain", useOffset=False, axis="both")

attained_plot = diagnostic_df.groupby(["Sex", "Attained_Age_mod"], dropna=False)[plot_metric_cols].sum().reset_index()
attained_plot = attained_plot[attained_plot[MODEL_Z_WEIGHT_COL] >= MIN_Z_WEIGHT_FOR_MONOTONICITY].copy()
attained_plot["Actual_Rate_per_1000"] = attained_plot[MODEL_ACTUAL_COL] * 1000 / attained_plot[MODEL_EXPOSURE_COL]
attained_plot["Expected_Rate_per_1000"] = attained_plot[MODEL_EXPECTED_COL] * 1000 / attained_plot[MODEL_EXPOSURE_COL]
attained_plot["Predicted_Rate_per_1000"] = attained_plot[MODEL_PREDICTED_COL] * 1000 / attained_plot[MODEL_EXPOSURE_COL]
attained_plot["Actual_to_Predicted"] = attained_plot["Actual_Rate_per_1000"] / attained_plot["Predicted_Rate_per_1000"]
attained_plot["Predicted_to_Expected"] = attained_plot["Predicted_Rate_per_1000"] / attained_plot["Expected_Rate_per_1000"]

for sex_value, part in attained_plot.groupby("Sex", dropna=False):
    part = part.sort_values("Attained_Age_mod")
    axes[1].plot(part["Attained_Age_mod"], part["Actual_Rate_per_1000"], marker="o", linestyle="-", label=f"{sex_value} | Actual")
    axes[1].plot(part["Attained_Age_mod"], part["Predicted_Rate_per_1000"], marker="s", linestyle="--", label=f"{sex_value} | Predicted")
axes[1].set_title(f"Actual vs predicted {MODEL_TARGET_LABEL.lower()} rate by Attained_Age_mod split by Sex")
axes[1].set_xlabel("Attained_Age_mod")
axes[1].set_ylabel(f"Rate per 1000 {MODEL_RATE_DENOMINATOR_LABEL}")
axes[1].grid(True, alpha=0.3)
axes[1].legend(title="Sex", fontsize=8, title_fontsize=9)
axes[1].ticklabel_format(style="plain", useOffset=False, axis="both")

for sex_value, part in attained_plot.groupby("Sex", dropna=False):
    part = part.sort_values("Attained_Age_mod")
    axes[2].plot(part["Attained_Age_mod"], part["Actual_to_Predicted"], marker="o", label=sex_value)
axes[2].axhline(1.0, linestyle="--", color="black", linewidth=1)
axes[2].set_title("A/P by Attained_Age_mod split by Sex")
axes[2].set_xlabel("Attained_Age_mod")
axes[2].set_ylabel("A/P")
axes[2].grid(True, alpha=0.3)
axes[2].legend(title="Sex", fontsize=8, title_fontsize=9)
axes[2].ticklabel_format(style="plain", useOffset=False, axis="both")

for sex_value, part in attained_plot.groupby("Sex", dropna=False):
    part = part.sort_values("Attained_Age_mod")
    axes[3].plot(part["Attained_Age_mod"], part["Predicted_to_Expected"], marker="o", label=sex_value)
axes[3].axhline(1.0, linestyle="--", color="black", linewidth=1)
axes[3].set_title("P/E by Attained_Age_mod split by Sex")
axes[3].set_xlabel("Attained_Age_mod")
axes[3].set_ylabel("P/E")
axes[3].grid(True, alpha=0.3)
axes[3].legend(title="Sex", fontsize=8, title_fontsize=9)
axes[3].ticklabel_format(style="plain", useOffset=False, axis="both")

year_plot = diagnostic_df.groupby(["Sex", "Observation_Year"], dropna=False)[plot_metric_cols].sum().reset_index()
year_plot = year_plot[year_plot[MODEL_Z_WEIGHT_COL] >= MIN_Z_WEIGHT_FOR_MONOTONICITY].copy()
year_plot["Predicted_Rate_per_1000"] = year_plot[MODEL_PREDICTED_COL] * 1000 / year_plot[MODEL_EXPOSURE_COL]
for sex_value, part in year_plot.groupby("Sex", dropna=False):
    part = part.sort_values("Observation_Year")
    axes[4].plot(part["Observation_Year"], part["Predicted_Rate_per_1000"], marker="s", linestyle="--", label=f"{sex_value} | Predicted")
axes[4].set_title(f"Predicted {MODEL_TARGET_LABEL.lower()} rate by Observation_Year split by Sex")
axes[4].set_xlabel("Observation_Year")
axes[4].set_ylabel(f"Rate per 1000 {MODEL_RATE_DENOMINATOR_LABEL}")
axes[4].grid(True, alpha=0.3)
axes[4].legend(title="Sex", fontsize=8, title_fontsize=9)
axes[4].ticklabel_format(style="plain", useOffset=False, axis="both")

axes[5].set_axis_off()
plt.show()




In [ ]:
# Cell 28: summarize model metrics
print("Final refit model:", final_model["name"])
display(final_metrics)
model_results_df = pd.DataFrame(model_results).drop_duplicates(subset=["Model"], keep="last").reset_index(drop=True)
display(model_results_df)


In [ ]:
# Cell 29: append last refit prediction to full.parquet
prediction_output_cols = {
    "count": "Predicted_Death_Count",
    "amount": "Predicted_Death_Amount",
}
predicted_parquet_col = prediction_output_cols.get(PREDICTION_BASIS, MODEL_PREDICTED_COL)
full_parquet_path = DATA_PATH
tmp_parquet_path = full_parquet_path.with_name(
    f"{full_parquet_path.stem}__tmp_with_prediction{full_parquet_path.suffix}"
)

if not final_model or "fit" not in final_model:
    raise RuntimeError("Run the full-scope refit cell before updating full.parquet.")

full_parquet_df = pd.read_parquet(full_parquet_path)
full_parquet_df = full_parquet_df.drop(columns=[predicted_parquet_col], errors="ignore")

full_score_df = full_parquet_df.copy()
for col in [
    "Observation_Year", "Issue_Age", "Duration", "Issue_Year", "Attained_Age",
    "Amount_Exposed", "Policies_Exposed", "Death_Claim_Amount", "Death_Count",
    "ExpDth_VBT2015_Cnt", "ExpDth_VBT2015_Amt",
]:
    if col in full_score_df.columns:
        full_score_df[col] = pd.to_numeric(full_score_df[col], errors="coerce")

full_score_df["Age_Ind"] = (
    full_score_df["Age_Ind"]
    .astype("string")
    .str.upper()
    .map({"ANB": "0", "ALB": "1", "0": "0", "1": "1"})
    .fillna(full_score_df["Age_Ind"].astype("string"))
)
full_score_df["Sex"] = (
    full_score_df["Sex"]
    .astype("string")
    .str.upper()
    .map({"MALE": "M", "FEMALE": "F", "M": "M", "F": "F"})
    .fillna(full_score_df["Sex"].astype("string"))
)

if PREDICTION_BASIS == "amount":
    cap = full_score_df["Face_Amount_Band"].isin([
        "08: 1,000,000 - 2,499,999",
        "09: 2,500,000 - 4,999,999",
        "10: 5,000,000 - 9,999,999",
        "11: 10,000,000+",
    ])
    full_score_df.loc[cap, "Death_Claim_Amount"] = 1_000_000 * full_score_df.loc[cap, "Death_Count"]
    full_score_df.loc[cap, "Amount_Exposed"] = 1_000_000 * full_score_df.loc[cap, "Policies_Exposed"]
    full_score_df.loc[cap, "ExpDth_VBT2015_Amt"] = 1_000_000 * full_score_df.loc[cap, "ExpDth_VBT2015_Cnt"]

full_score_df[ISSUE_AGE_MOD_COL] = full_score_df["Issue_Age"]
full_score_df[ATTAINED_AGE_MOD_COL] = full_score_df["Attained_Age"]
alb_mask = full_score_df["Age_Ind"].astype("string") == "1"
full_score_df.loc[alb_mask, ISSUE_AGE_MOD_COL] = full_score_df.loc[alb_mask, "Issue_Age"] + 0.5
full_score_df.loc[alb_mask, ATTAINED_AGE_MOD_COL] = full_score_df.loc[alb_mask, "Attained_Age"] + 0.5

issue_year = pd.to_numeric(full_score_df["Issue_Year"], errors="coerce")
scale = np.where(issue_year.between(2014, 2019), 7 / (2019 - issue_year + 1).clip(lower=1), 1.0)
full_score_df["Policies_Exposed"] = full_score_df["Policies_Exposed"] * scale
full_score_df["Amount_Exposed"] = full_score_df["Amount_Exposed"] * scale

full_score_df["Issue_Age_mod_spline"] = full_score_df[ISSUE_AGE_MOD_COL].clip(
    np.nextafter(ISSUE_AGE_MIN, np.inf), np.nextafter(ISSUE_AGE_MOD_MAX, -np.inf)
)
full_score_df["Attained_Age_mod_spline"] = full_score_df[ATTAINED_AGE_MOD_COL].clip(
    np.nextafter(ATTAINED_AGE_MIN, np.inf), np.nextafter(ATTAINED_AGE_MOD_MAX, -np.inf)
)
full_score_df["Sex_M"] = (full_score_df["Sex"].astype(str) == "M").astype(float)
full_score_df["Sex_x_Attained_Age_mod"] = full_score_df["Sex_M"] * full_score_df[ATTAINED_AGE_MOD_COL]

score_mask = (
    full_score_df["Sex"].isin(["M", "F"])
    & pd.to_numeric(full_score_df[MODEL_EXPOSURE_COL], errors="coerce").gt(0)
    & pd.to_numeric(full_score_df[ATTAINED_AGE_MOD_COL], errors="coerce").notna()
).fillna(False)

score_mask_values = score_mask.to_numpy(dtype=bool)
predicted_values_full = np.full(len(full_parquet_df), np.nan, dtype=np.float32)
score_rows = full_score_df.loc[score_mask].copy()

def predict_refit_model(model, rows):
    exposure = pd.to_numeric(rows[MODEL_EXPOSURE_COL], errors="coerce").clip(lower=1e-10)
    offset = np.log(exposure)
    kind = model["kind"]

    if kind == "penalized_combined_gam":
        return model["fit"]["wrapper"].predict(rows, offset=offset.values)
    if kind == "statsmodels_formula":
        return model["fit"].predict(rows, offset=offset)
    if kind == "per_sex_formula":
        pred = np.full(len(rows), np.nan, dtype=float)
        for sex_value, fit in model["fit"].items():
            mask = rows["Sex"].astype(str).to_numpy() == str(sex_value)
            if mask.any():
                pred[mask] = fit.predict(rows.loc[mask], offset=offset.loc[rows.index[mask]])
        return pred
    if kind == "wh":
        pred = np.full(len(rows), np.nan, dtype=float)
        for sex_value, fit in model["fit"].items():
            age, rate = fit
            mask = rows["Sex"].astype(str).to_numpy() == str(sex_value)
            if mask.any():
                pred[mask] = np.interp(rows.loc[mask, ATTAINED_AGE_MOD_COL], age, rate) * exposure.loc[rows.index[mask]]
        return pred
    if kind == "pygam":
        columns = model.get("columns", ML_FEATURES)
        return model["fit"].predict(rows[columns].to_numpy(dtype=float), exposure=exposure)
    if kind == "sklearn_rate":
        columns = model.get("columns", ML_FEATURES)
        return model["fit"].predict(rows[columns]) * exposure.to_numpy(dtype=float)
    if kind == "sklearn_log_rate":
        columns = model.get("columns", ML_FEATURES)
        return np.exp(model["fit"].predict(rows[columns])) * exposure.to_numpy(dtype=float)
    raise ValueError(f"Unsupported final_model kind: {kind}")

if len(score_rows) > 0:
    predicted_values_full[score_mask_values] = np.clip(
        predict_refit_model(final_model, score_rows),
        1e-12,
        None,
    ).astype(np.float32)

full_parquet_df[predicted_parquet_col] = predicted_values_full
original_size_mb = full_parquet_path.stat().st_size / (1024 ** 2)
if tmp_parquet_path.exists():
    tmp_parquet_path.unlink()

try:
    import pyarrow as pa
    parquet_compression = "zstd" if pa.Codec.is_available("zstd") else "snappy"
    parquet_kwargs = {"engine": "pyarrow", "index": False, "compression": parquet_compression}
    if parquet_compression == "zstd":
        try:
            full_parquet_df.to_parquet(tmp_parquet_path, compression_level=9, **parquet_kwargs)
        except (TypeError, ValueError):
            full_parquet_df.to_parquet(tmp_parquet_path, **parquet_kwargs)
    else:
        full_parquet_df.to_parquet(tmp_parquet_path, **parquet_kwargs)
except ImportError:
    parquet_compression = "snappy"
    full_parquet_df.to_parquet(tmp_parquet_path, index=False, compression=parquet_compression)

new_size_mb = tmp_parquet_path.stat().st_size / (1024 ** 2)
tmp_parquet_path.replace(full_parquet_path)
print(
    f"Updated {full_parquet_path} with {predicted_parquet_col} from {final_model['name']} "
    f"({PREDICTION_BASIS}); scored {int(score_mask_values.sum()):,} of {len(full_parquet_df):,} rows. "
    f"Size: {original_size_mb:.1f} MB -> {new_size_mb:.1f} MB ({new_size_mb - original_size_mb:+.1f} MB)."
)
